# Kiểm tra #1: Python cơ bản
**Lập trình phân tích dữ liệu**

Dữ liệu: tập tin `Data.xlsx` gồm 2 trang tính:
- **Users**: `userID`, `action`, `drama`, `animation`, `documentation` (mức độ ưa thích của người dùng, trong [0,1])
- **Films**: `filmID`, `action`, `drama`, `animation`, `documentation` (mức độ liên quan của phim đến từng thể loại, trong [0,1])

> **Lưu ý về cách tính độ phù hợp:** vì đề không cho công thức cụ thể, bài làm dùng công thức phổ biến cho bài toán content-based recommendation:
> điểm phù hợp giữa người dùng *u* và phim *f* (theo một/nhiều thể loại) = tổng **(mức độ ưa thích của u đối với thể loại g) × (mức độ liên quan của f đối với thể loại g)**.
> Nếu giảng viên yêu cầu công thức khác, chỉ cần thay công thức tính `score` trong các hàm bên dưới, phần còn lại (nhập liệu, kiểm tra hợp lệ, sắp xếp, hiển thị) không cần đổi.

In [1]:
import pandas as pd

# Đường dẫn tới tập tin dữ liệu (đặt Data.xlsx cùng thư mục với file .ipynb này)
FILE_NAME = "Data.xlsx"

users_df = pd.read_excel(FILE_NAME, sheet_name="Users")
films_df = pd.read_excel(FILE_NAME, sheet_name="Films")

# Chuẩn hóa kiểu dữ liệu mã số về string để so sánh khi nhập liệu không bị lệch kiểu
users_df["userID"] = users_df["userID"].astype(str)
films_df["filmID"] = films_df["filmID"].astype(str)

GENRES = ["action", "drama", "animation", "documentation"]

print("Users:")
display(users_df.head())
print("Films:")
display(films_df.head())

Users:


,userID,action,drama,animation,documentation
0,u1,0.2,0.8,0.6,0.2
1,u2,0.8,0.2,0.4,0.8
2,u3,0.4,0.4,0.6,0.6
3,u4,0.8,0.7,0.4,0.7
4,u5,0.4,0.9,0.7,0.3


Films:


,filmID,action,drama,animation,documentation
0,f1,0.3,0.8,0.6,0.3
1,f2,0.7,0.2,0.4,0.7
2,f3,0.1,0.2,0.5,0.7
3,f4,0.0,0.6,0.0,0.9
4,f5,0.4,0.9,0.7,0.3


## Yêu cầu 1
Nhập **mã số người sử dụng**, số nguyên **m ∈ [1,3]** và **thể loại phim**.
Liệt kê tối thiểu m bộ phim phù hợp nhất với người dùng theo đúng thể loại đã chọn
(nếu có nhiều phim đồng điểm ở vị trí thứ m thì lấy tất cả các phim đồng điểm đó, nên danh sách có thể nhiều hơn m).
Cho phép lặp lại nhiều lần; kiểm tra hợp lệ mã số người dùng và thể loại phim.

In [ ]:
def nhap_userID():
    """Nhập và kiểm tra hợp lệ mã số người sử dụng. Gõ 'stop' để dừng chức năng."""
    while True:
        user_id = input("Nhập mã số người sử dụng (gõ 'stop' để thoát chức năng): ").strip()
        if user_id.lower() == "stop":
            return None
        if user_id in users_df["userID"].values:
            return user_id
        print("Mã số người sử dụng không hợp lệ. Vui lòng nhập lại.")


def nhap_so_nguyen(ten_bien, gia_tri_min, gia_tri_max):
    """Nhập và kiểm tra một số nguyên nằm trong [gia_tri_min, gia_tri_max]."""
    while True:
        chuoi = input(f"Nhập số nguyên {ten_bien} ({gia_tri_min} <= {ten_bien} <= {gia_tri_max}): ").strip()
        try:
            gia_tri = int(chuoi)
            if gia_tri_min <= gia_tri <= gia_tri_max:
                return gia_tri
            print(f"{ten_bien} phải nằm trong khoảng [{gia_tri_min}, {gia_tri_max}]. Vui lòng nhập lại.")
        except ValueError:
            print("Vui lòng nhập một số nguyên hợp lệ.")


def nhap_the_loai():
    """Nhập và kiểm tra hợp lệ thể loại phim."""
    while True:
        genre = input(f"Nhập thể loại phim {GENRES}: ").strip().lower()
        if genre in GENRES:
            return genre
        print("Thể loại phim không hợp lệ. Vui lòng nhập lại.")


def lay_it_nhat_k_dong_dau(df_da_sap_xep, cot_diem, k):
    """Trả về ít nhất k dòng đầu của df đã sắp xếp giảm dần theo cot_diem
    (lấy thêm các dòng đồng điểm ở vị trí thứ k)."""
    if len(df_da_sap_xep) <= k:
        return df_da_sap_xep
    diem_cutoff = df_da_sap_xep.iloc[k - 1][cot_diem]
    return df_da_sap_xep[df_da_sap_xep[cot_diem] >= diem_cutoff]


def goi_y_phim_cho_nguoi_dung():
    """Yêu cầu 1: gợi ý phim phù hợp cho người dùng theo một thể loại cụ thể."""
    while True:
        user_id = nhap_userID()
        if user_id is None:
            return
        m = nhap_so_nguyen("m", 1, 3)
        genre = nhap_the_loai()

        # Mức độ ưa thích của người dùng đối với thể loại đã chọn
        user_pref = users_df.loc[users_df["userID"] == user_id, genre].iloc[0]

        # Điểm phù hợp = mức độ ưa thích của người dùng * mức độ liên quan của phim đến thể loại
        films_score = films_df.copy()
        films_score["score"] = user_pref * films_score[genre]
        films_score = films_score.sort_values(by="score", ascending=False)

        ket_qua = lay_it_nhat_k_dong_dau(films_score, "score", m)

        print(f"\n Ít nhất {m} phim thuộc thể loại '{genre}' phù hợp nhất với người dùng {user_id}:")
        print(ket_qua[["filmID", genre, "score"]].to_string(index=False))

        tiep_tuc = input("\nThực hiện lại chức năng này? (y/n): ").strip().lower()
        if tiep_tuc != "y":
            break


goi_y_phim_cho_nguoi_dung()

Mã số người sử dụng không hợp lệ. Vui lòng nhập lại.


## Yêu cầu 2
Nhập **mã số bộ phim** và số nguyên **n ∈ [1,3]**.
Liệt kê tối thiểu n người sử dụng có thể ưa thích bộ phim này nhất
(dựa trên tổng mức độ phù hợp trên cả 4 thể loại giữa mỗi người dùng và bộ phim).
Cho phép lặp lại nhiều lần; kiểm tra hợp lệ mã số bộ phim.

In [ ]:
def nhap_filmID():
    """Nhập và kiểm tra hợp lệ mã số bộ phim. Gõ 'stop' để dừng chức năng."""
    while True:
        film_id = input("Nhập mã số bộ phim (gõ 'stop' để thoát chức năng): ").strip()
        if film_id.lower() == "stop":
            return None
        if film_id in films_df["filmID"].values:
            return film_id
        print(">> Mã số bộ phim không hợp lệ. Vui lòng nhập lại.")


def goi_y_nguoi_dung_cho_phim():
    """Yêu cầu 2: gợi ý người dùng có thể thích một bộ phim cụ thể nhất."""
    while True:
        film_id = nhap_filmID()
        if film_id is None:
            return
        n = nhap_so_nguyen("n", 1, 3)

        film_row = films_df.loc[films_df["filmID"] == film_id].iloc[0]

        # Điểm phù hợp giữa từng người dùng và phim = tổng trên 4 thể loại của
        # (mức độ ưa thích thể loại của người dùng * mức độ liên quan thể loại của phim)
        users_score = users_df.copy()
        users_score["score"] = sum(users_score[g] * film_row[g] for g in GENRES)
        users_score = users_score.sort_values(by="score", ascending=False)

        ket_qua = lay_it_nhat_k_dong_dau(users_score, "score", n)

        print(f"\n>> Ít nhất {n} người dùng có thể ưa thích phim {film_id} nhất:")
        print(ket_qua[["userID", "score"]].to_string(index=False))

        tiep_tuc = input("\nThực hiện lại chức năng này? (y/n): ").strip().lower()
        if tiep_tuc != "y":
            break


goi_y_nguoi_dung_cho_phim()

## Yêu cầu 3
Bộ phim nào có thể được ưa thích nhiều nhất — tính trên toàn bộ người dùng
(tổng điểm phù hợp của một phim với tất cả người dùng, phim có tổng điểm cao nhất là phim được ưa thích nhiều nhất).

In [ ]:
def phim_duoc_ua_thich_nhat():
    """Yêu cầu 3: phim có tổng điểm phù hợp với tất cả người dùng cao nhất."""
    tong_ua_thich_theo_the_loai = {g: users_df[g].sum() for g in GENRES}

    films_total = films_df.copy()
    films_total["total_score"] = sum(
        tong_ua_thich_theo_the_loai[g] * films_total[g] for g in GENRES
    )
    films_total = films_total.sort_values(by="total_score", ascending=False)

    phim_tot_nhat = films_total.iloc[0]
    print(">> Bảng tổng điểm phù hợp của từng phim (5 phim đầu):")
    print(films_total[["filmID", "total_score"]].head().to_string(index=False))
    print(f"\n>> Bộ phim được ưa thích nhiều nhất: {phim_tot_nhat['filmID']} "
          f"(tổng điểm phù hợp = {phim_tot_nhat['total_score']:.4f})")


phim_duoc_ua_thich_nhat()

## Yêu cầu 4
Thể loại phim nào có thể được ưa thích nhiều nhất — dựa trên tổng mức độ ưa thích
của tất cả người dùng đối với từng thể loại.

In [ ]:
def the_loai_duoc_ua_thich_nhat():
    """Yêu cầu 4: thể loại có tổng mức độ ưa thích của tất cả người dùng cao nhất."""
    tong_theo_the_loai = {g: users_df[g].sum() for g in GENRES}
    the_loai_tot_nhat = max(tong_theo_the_loai, key=tong_theo_the_loai.get)

    print(">> Tổng mức độ ưa thích của tất cả người dùng theo từng thể loại:")
    for g, tong in tong_theo_the_loai.items():
        print(f"   {g:15s}: {tong:.4f}")
    print(f"\n>> Thể loại phim được ưa thích nhiều nhất: {the_loai_tot_nhat}")


the_loai_duoc_ua_thich_nhat()

## Chương trình chính (menu)
Chạy ô bên dưới để có một menu gọi lại cả 4 chức năng trên trong cùng một vòng lặp.

In [ ]:
def main():
    chuc_nang = {
        "1": goi_y_phim_cho_nguoi_dung,
        "2": goi_y_nguoi_dung_cho_phim,
        "3": phim_duoc_ua_thich_nhat,
        "4": the_loai_duoc_ua_thich_nhat,
    }
    while True:
        print("\n===== MENU =====")
        print("1. Gợi ý phim phù hợp cho người dùng theo thể loại (Yêu cầu 1)")
        print("2. Gợi ý người dùng có thể thích một bộ phim (Yêu cầu 2)")
        print("3. Xem bộ phim được ưa thích nhiều nhất (Yêu cầu 3)")
        print("4. Xem thể loại phim được ưa thích nhiều nhất (Yêu cầu 4)")
        print("0. Thoát chương trình")
        lua_chon = input("Chọn chức năng (0-4): ").strip()

        if lua_chon == "0":
            print("Kết thúc chương trình.")
            break
        elif lua_chon in chuc_nang:
            chuc_nang[lua_chon]()
        else:
            print(">> Lựa chọn không hợp lệ. Vui lòng chọn lại.")


# Bỏ dấu # ở dòng dưới để chạy menu tổng (gọi lần lượt các chức năng theo lựa chọn)
# main()